In [1]:
# ============================================
# DEEPFAKE DETECTION - 3-DATASET T=32 TRAINING
# Experiment A1: Full Model (adaptive fusion)
# ============================================
# Datasets: DFDC02 + DFD01 + CelebDF (all preprocessed with T=32)
# GPU: P100 recommended (16GB)
#
# Required Kaggle inputs:
#   - dfdc02-preprocessed-t32
#   - dfd01-preprocessed-t32
#   - celebdf-preprocessed-t32
#
# This notebook creates the split file and saves it in output
# for A2/A3/A4 notebooks to reuse.

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Write training script
train_script = r'''
import os, sys, time, json, gc, shutil
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f'numpy version: {np.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
assert torch.cuda.is_available(), 'No GPU detected!'
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# =============================================
# Find ALL T=32 datasets in /kaggle/input
# =============================================
TARGET_T = 32
print(f'\n=== Searching for T={TARGET_T} datasets ===')
found_datasets = []
for root, dirs, files in os.walk('/kaggle/input'):
    dirname_lower = os.path.basename(root).lower()
    if '_16' in dirname_lower or '-16' in dirname_lower:
        continue
    # Skip result datasets
    if '3ds-t32-a' in dirname_lower.replace('_', '-'):
        continue

    if 'real' in dirs and 'fake' in dirs:
        real_p = os.path.join(root, 'real')
        fake_p = os.path.join(root, 'fake')
        rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
        fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
        if rc > 0 and fc > 0:
            sample_dir = None
            for label in ('real', 'fake'):
                lp = os.path.join(root, label)
                for vd in os.listdir(lp):
                    vdp = os.path.join(lp, vd)
                    if os.path.isdir(vdp):
                        sample_dir = vdp
                        break
                if sample_dir:
                    break
            if sample_dir:
                n_frames = len([f for f in os.listdir(sample_dir) if f.endswith('.jpg')])
                if n_frames > 0 and abs(n_frames - TARGET_T) > TARGET_T * 0.5:
                    print(f'  SKIP: {root} (frames={n_frames}, expected ~{TARGET_T})')
                    continue

            path_lower = root.lower()
            if 'dfd01' in path_lower or 'dfd-01' in path_lower or 'dfd_01' in path_lower:
                ds_name = 'dfd01'
            elif 'dfdc02' in path_lower or 'dfdc-02' in path_lower or 'dfdc' in path_lower:
                ds_name = 'dfdc02'
            elif 'celeb' in path_lower:
                ds_name = 'celebdf'
            else:
                ds_name = os.path.basename(root).replace('-', '_')

            if any(d['name'] == ds_name for d in found_datasets):
                print(f'  SKIP duplicate: {ds_name} at {root}')
                continue

            found_datasets.append({'path': root, 'name': ds_name, 'real': rc, 'fake': fc})
            print(f'  Found: {ds_name} at {root} (real={rc}, fake={fc})')

if len(found_datasets) < 3:
    print(f'WARNING: Expected 3 datasets, found {len(found_datasets)}!')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    if len(found_datasets) == 0:
        sys.exit(1)

dataset_roots = '+'.join(d['path'] for d in found_datasets)
dataset_names = '+'.join(d['name'] for d in found_datasets)
total_videos = sum(d['real'] + d['fake'] for d in found_datasets)
print(f'\nCombined: {dataset_names}')
print(f'Total videos: {total_videos}')

# =============================================
# Run experiment A1: Full Model
# =============================================
from config import Config
from train import train
from utils import save_metrics

SPLIT_FILENAME = 'split_seed42_3ds_t32.json'
EXP_NAME = 'A1_full_model'

print(f'\n{"="*70}')
print(f'[1/1] {EXP_NAME} (model_type=full, fusion=adaptive, T=32, 3-dataset)')
print(f'{"="*70}\n')

cfg = Config()
cfg.dataset_root = dataset_roots
cfg.dataset_name = dataset_names
cfg.model_type = 'full'
cfg.fusion_type = 'adaptive'
cfg.num_frames = 32
cfg.min_frames_per_video = 24
cfg.max_epochs = 30
cfg.batch_size = 8
cfg.patience = 7
cfg.device = 'auto'
cfg.use_amp = True
cfg.num_workers = 2
cfg.pin_memory = True
cfg.splits_dir = './splits'
cfg.output_dir = './experiments'
cfg.seed = 42
cfg.unfreeze_last_n_blocks = 2
cfg.split_filename = SPLIT_FILENAME
cfg.split_mode = 'random'
cfg.save_split = True

t0 = time.time()
all_results = []
try:
    metrics = train(cfg)
    metrics['experiment_name'] = EXP_NAME
    metrics['status'] = 'success'
    metrics['duration_min'] = round((time.time() - t0) / 60, 1)
    all_results.append(metrics)
    test = metrics.get('test', {})
    print(f'\n[OK] {EXP_NAME} done in {metrics["duration_min"]} min')
    print(f'     AUC={test.get("auc",0):.4f}  Acc={test.get("accuracy",0):.4f}  F1={test.get("f1",0):.4f}')
except Exception as e:
    elapsed = round((time.time() - t0) / 60, 1)
    print(f'\n[FAIL] {EXP_NAME} after {elapsed} min: {e}')
    import traceback; traceback.print_exc()
    all_results.append({
        'experiment_name': EXP_NAME,
        'status': 'failed',
        'error': str(e),
        'duration_min': elapsed,
    })

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

save_metrics(all_results, './experiments/all_results_3ds_t32_a1.json')

# =============================================
# Summary
# =============================================
print(f'\n{"="*70}')
print('RESULTS SUMMARY - A1 Full Model (DFDC02+DFD01+CelebDF, T=32)')
print(f'{"="*70}')
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        be = r.get('best_epoch', '?')
        print(f'{r["experiment_name"]:<25} AUC={t.get("auc",0):.4f}  Acc={t.get("accuracy",0):.4f}  '
              f'F1={t.get("f1",0):.4f}  EER={t.get("eer",0):.4f}  Epoch={be}  Time={r.get("duration_min",0):.1f}m')
    else:
        print(f'{r["experiment_name"]:<25} FAILED: {r.get("error","")[:60]}')

# Save results + splits to /kaggle/working/
os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.makedirs('/kaggle/working/splits', exist_ok=True)
os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
os.system('cp -r splits/ /kaggle/working/splits/ 2>/dev/null')
os.system('tar czf /kaggle/working/results_3ds_t32_a1.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
print(f'\nresults_3ds_t32_a1.tar.gz ready for download')
print(f'Split file saved in /kaggle/working/splits/ for A2/A3/A4 reuse')
'''

with open('/kaggle/working/run_training_3ds_t32_a1.py', 'w') as f:
    f.write(train_script)

# Step 4: Run training
print('\n=== Starting A1 Full Model Training (T=32, 3-dataset) ===')
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_training_3ds_t32_a1.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f'\nTraining subprocess exited with code: {result.returncode}')

if os.path.exists('/kaggle/working/project/experiments'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/experiments/', '/kaggle/working/experiments/'])
    print('Results copied to /kaggle/working/experiments/')
if os.path.exists('/kaggle/working/project/splits'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/splits/', '/kaggle/working/splits/'])
    print('Splits copied to /kaggle/working/splits/')
if os.path.exists('/kaggle/working/results_3ds_t32_a1.tar.gz'):
    print('results_3ds_t32_a1.tar.gz is ready for download')

=== Installing dependencies ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 57.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

Dependencies installed.


...

Project cloned.

=== Starting A1 Full Model Training (T=32, 3-dataset) ===


2026-04-03 18:35:32 | INFO | ======================================================================
2026-04-03 18:35:32 | INFO | Эксперимент: celebdf+dfdc02+dfd01_full_seed42_bs8_T32_adaptive
2026-04-03 18:35:32 | INFO | Dataset: celebdf+dfdc02+dfd01
2026-04-03 18:35:32 | INFO | Model: full
2026-04-03 18:35:32 | INFO | Fusion: adaptive
2026-04-03 18:35:32 | INFO | Seed: 42
2026-04-03 18:35:32 | INFO | Device: cuda
2026-04-03 18:35:32 | INFO | Batch size: 8
2026-04-03 18:35:32 | INFO | Epochs: 30
2026-04-03 18:35:32 | INFO | Primary metric: auc
2026-04-03 18:35:32 | INFO | ======================================================================
2026-04-03 18:35:32 | INFO | Загрузка данных...
2026-04-03 18:36:47 | INFO | Train: 6966 videos | Val: 1492 | Test: 1495
2026-04-03 18:36:47 | INFO | Создание модели...
2026-04-03 18:36:52 | INFO | Warmup: spatial backbone заморожен (эпохи 1..5)
2026-04-03 18:36:52 | INFO | Параметры: всего=24,689,351, обучаемых=7,140,735
2026-04-03 18:36:52 | INFO

numpy version: 1.26.4
PyTorch: 2.2.2+cu121
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB

=== Searching for T=32 datasets ===
  Found: celebdf at /kaggle/input/datasets/alexandertarakanov/celebdf-preprocessed-t32 (real=589, fake=2710)
  Found: dfdc02 at /kaggle/input/datasets/alexandertarakanov/dfdc02-preprocessed-t32/preprocessed_DFDC02_32 (real=1699, fake=1541)
  Found: dfd01 at /kaggle/input/datasets/alexandertarakanov/dfd01-preprocessed-t32 (real=363, fake=3051)

Combined: celebdf+dfdc02+dfd01
Total videos: 9953

[1/1] A1_full_model (model_type=full, fusion=adaptive, T=32, 3-dataset)

[ДАННЫЕ] Всего видео: 3299 (real: 589, fake: 2710)
[ДАННЫЕ] Всего видео: 3240 (real: 1699, fake: 1541)
[ДАННЫЕ] Всего видео: 3414 (real: 363, fake: 3051)
[MULTI-DATA] Всего видео: 9953 (real: 2651, fake: 7302)
  [celebdf] 3299 видео (real: 589, fake: 2710)
  [dfdc02] 3240 видео (real: 1699, fake: 1541)
  [dfd01] 3414 видео (real: 363, fake: 3051)
[SPLIT] train: 6966, val: 1492, test: 1495


2026-04-03 18:38:22 | INFO |   [batch 87/870] avg_loss=0.7727
2026-04-03 18:39:50 | INFO |   [batch 174/870] avg_loss=0.7741
2026-04-03 18:41:21 | INFO |   [batch 261/870] avg_loss=0.7731
2026-04-03 18:42:49 | INFO |   [batch 348/870] avg_loss=0.7719
2026-04-03 18:44:18 | INFO |   [batch 435/870] avg_loss=0.7706
2026-04-03 18:45:52 | INFO |   [batch 522/870] avg_loss=0.7705
2026-04-03 18:47:25 | INFO |   [batch 609/870] avg_loss=0.7686
2026-04-03 18:48:56 | INFO |   [batch 696/870] avg_loss=0.7670
2026-04-03 18:50:28 | INFO |   [batch 783/870] avg_loss=0.7665
2026-04-03 18:52:01 | INFO |   [batch 870/870] avg_loss=0.7654
2026-04-03 18:54:47 | INFO | Epoch 01/30 | Train Loss: 0.7654 | Train AUC: 0.5207 | Val Loss: 0.7305 | Val AUC: 0.6192 | Val Acc: 0.4129 | Val F1: 0.3848 | LR backbone: 0.000020 | LR head: 0.000060 | Time: 1075.8s
2026-04-03 18:54:48 | INFO | Лучшая модель сохранена: auc = 0.6192
2026-04-03 18:56:02 | INFO |   [batch 87/870] avg_loss=0.6737
2026-04-03 18:57:15 | INFO |


[OK] A1_full_model done in 535.5 min
     AUC=0.9943  Acc=0.9672  F1=0.9776

RESULTS SUMMARY - A1 Full Model (DFDC02+DFD01+CelebDF, T=32)
A1_full_model             AUC=0.9943  Acc=0.9672  F1=0.9776  EER=0.0406  Epoch=28  Time=535.5m

results_3ds_t32_a1.tar.gz ready for download
Split file saved in /kaggle/working/splits/ for A2/A3/A4 reuse

Training subprocess exited with code: 0
Results copied to /kaggle/working/experiments/
Splits copied to /kaggle/working/splits/
results_3ds_t32_a1.tar.gz is ready for download
